In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import json
import gzip

def load_amazon_json_sample(file_path, max_lines=100000):
    data = []
    with gzip.open(file_path, "rt", encoding="utf-8") as f:  # Use gzip to open the file
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            data.append(json.loads(line))  # Parse each line as JSON
    return pd.DataFrame(data)

# Provide the correct path to the JSONL file on your Google Drive
df = load_amazon_json_sample("/content/drive/My Drive/meta_Electronics.jsonl (1).gz", max_lines=100000)

# Print the columns and first few rows
print("Columns:", df.columns.tolist())
print(df.head(3))


Columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']
     main_category                                              title  \
0  All Electronics             FS-1051 FATSHARK TELEPORTER V3 HEADSET   
1  All Electronics                      Ce-H22B12-S1 4Kx2K Hdmi 4Port   
2        Computers  Digi-Tatoo Decal Skin Compatible With MacBook ...   

   average_rating  rating_number  \
0             3.5              6   
1             5.0              1   
2             4.5            246   

                                            features  \
0                                                 []   
1             [UPC: 662774021904, Weight: 0.600 lbs]   
2  [WARNING: Please IDENTIFY MODEL NUMBER on the ...   

                                         description  price  \
0  [Teleporter V3 The “Teleporter V3” kit sets a ...   No

In [3]:
import pandas as pd
import json
import gzip

def load_amazon_json_sample(file_path, max_lines=100000):
    data = []
    with gzip.open(file_path, "rt", encoding="utf-8") as f:  # Use gzip to open the file
        for i, line in enumerate(f):
            if i >= max_lines:  # Stop if the maximum lines are reached
                break
            data.append(json.loads(line.strip()))  # Parse each line as JSON
    return pd.DataFrame(data)

# Provide the correct path to the JSONL file on your Google Drive
file_path = '/content/drive/My Drive/Electronics.jsonl (1).gz'
df = load_amazon_json_sample(file_path, max_lines=100000)

# Print the columns and first few rows
print("Columns:", df.columns.tolist())
print(df.head(3))


Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
   rating                                    title  \
0     3.0        Smells like gasoline! Going back!   
1     1.0  Didn’t work at all lenses loose/broken.   
2     5.0                               Excellent!   

                                                text  \
0  First & most offensive: they reek of gasoline ...   
1  These didn’t work. Idk if they were damaged in...   
2  I love these. They even come with a carry case...   

                                              images        asin parent_asin  \
0  [{'small_image_url': 'https://m.media-amazon.c...  B083NRGZMM  B083NRGZMM   
1                                                 []  B07N69T6TM  B07N69T6TM   
2                                                 []  B01G8JO5F2  B01G8JO5F2   

                        user_id      timestamp  helpful_vote  \
0  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  165818511

In [4]:
import pandas as pd
import json
import gzip

# Function to load reviews (ensure it's loaded properly)
def load_amazon_json_sample(file_path, max_lines=100000):
    data = []
    with gzip.open(file_path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            data.append(json.loads(line.strip()))
    return pd.DataFrame(data)

# Load reviews data
reviews_path = "/content/drive/My Drive/Electronics.jsonl (1).gz"  # Your reviews path
df_reviews = load_amazon_json_sample(reviews_path, max_lines=100000)

# Load metadata (item details)
metadata_path = "/content/drive/My Drive/meta_Electronics.jsonl (1).gz"  # Your metadata path
df_metadata = load_amazon_json_sample(metadata_path, max_lines=100000)

# Merge the two dataframes on 'parent_asin'
merged_df = pd.merge(df_reviews, df_metadata, on='parent_asin', how='inner')

# Inspect the result
print(merged_df.head())


   rating                                       title_x  \
0     4.0                                    Four Stars   
1     5.0                       Nice lightweight sleeve   
2     4.0  Did anyone use this for a recessed TV mount?   
3     5.0                            Easy and plentiful   
4     5.0                                         Great   

                                                text images_x        asin  \
0           Pretty cool!!!<br /><br />Thanks Amazon.       []  B00ZV9RDKK   
1  I am not sure how much protection this provide...       []  B01976K5B6   
2  I believe the electrical boxes are made especi...       []  B001PL3XJS   
3  Stick great to my desk and provide ample space...       []  B00V3KM73E   
4                                      Fast charging       []  B07PGRR6QN   

  parent_asin                       user_id      timestamp  helpful_vote  \
0  B075X8471B  AEM663T6XHZFWLODF4US2RCOCUSA  1509491262120             0   
1  B08P3VMW76  AGBFYI2DDIKXC5Y

In [2]:
import pandas as pd
import numpy as np
from openai import OpenAI
from transformers import pipeline
from concurrent.futures import ThreadPoolExecutor
import time
import os

# 1. Initialize Services
print("Initializing services...")
start_time = time.time()

# Initialize OpenAI API client
client = OpenAI(api_key="your_openai_api_key", timeout=15.0, max_retries=3)

# Load sentiment model
sentiment = pipeline("sentiment-analysis",
                    model="distilbert-base-uncased-finetuned-sst-2-english",
                    device=0 if os.environ.get('CUDA_VISIBLE_DEVICES') else -1,
                    batch_size=64)

print(f"Services initialized in {time.time()-start_time:.2f} seconds")

# 2. Load and Sample Data
print("\nLoading and sampling data...")
data_start = time.time()

def safe_sample(df, sample_size=50000):
    """Safe sampling with size validation"""
    sample_size = min(sample_size, len(df))
    return df.sample(n=sample_size, random_state=42)

# Load your dataframe here
# merged_df = pd.read_csv('your_data.csv')
merged_df = safe_sample(merged_df)

# Validate required columns
required_columns = ['title', 'text', 'average_rating', 'rating_number', 'asin', 'main_category']
missing_cols = [col for col in required_columns if col not in merged_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"Data loaded and sampled in {time.time()-data_start:.2f} seconds")
print(f"Working with {len(merged_df)} records")

# 3. Sentiment Analysis
print("\nStarting sentiment analysis...")
sentiment_start = time.time()

def batch_sentiment(texts):
    """Process texts in batches with error handling"""
    try:
        results = sentiment(list(texts))
        return [{'label': r['label'], 'score': r['score']} for r in results]
    except Exception:
        return [{'label': 'NEUTRAL', 'score': 0.5} for _ in texts]

def parallel_sentiment(df, batch_size=2000):
    """Parallel sentiment analysis with progress tracking"""
    texts = df['text'].fillna('').astype(str).tolist()
    results = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            futures.append(executor.submit(batch_sentiment, batch))

        for i, future in enumerate(futures):
            results.extend(future.result())
            if (i+1) % 5 == 0:  # Print progress every 5 batches
                print(f"Processed {(i+1)*batch_size:,} reviews...")

    return results

sentiment_results = parallel_sentiment(merged_df)
merged_df['sentiment'] = [x['label'] for x in sentiment_results]
merged_df['sentiment_score'] = [x['score'] for x in sentiment_results]

print(f"Sentiment analysis completed in {time.time()-sentiment_start:.2f} seconds")

# 4. Generate Insights
print("\nGenerating product insights...")
insights_start = time.time()

def generate_insight(row):
    """Generate insight based on rating and sentiment"""
    text_snippet = f"{row['title']}: {str(row['text'])[:100]}..." if pd.notna(row['text']) else ""

    if row['average_rating'] >= 4 or (row['sentiment'] == 'POSITIVE' and row['sentiment_score'] > 0.8):
        return {'type': 'praise', 'text': text_snippet}
    elif row['average_rating'] <= 2 or (row['sentiment'] == 'NEGATIVE' and row['sentiment_score'] > 0.8):
        return {'type': 'complaint', 'text': text_snippet}
    return None

# Apply insights generation
insights = merged_df.apply(generate_insight, axis=1)

# Organize insights by product
product_insights = merged_df.groupby('asin').apply(
    lambda g: {
        'praises': [i['text'] for i in insights[g.index] if i and i['type'] == 'praise'][:3],
        'complaints': [i['text'] for i in insights[g.index] if i and i['type'] == 'complaint'][:3]
    }
)

# Handle potential duplicate column
if 'insights' in merged_df.columns:
    merged_df = merged_df.drop(columns=['insights'])

# Merge insights back
merged_df = merged_df.merge(
    product_insights.reset_index(name='insights'),
    on='asin',
    how='left'
)

# Fill empty insights
default_insights = {'praises': ["No detailed praises"], 'complaints': ["No major complaints"]}
merged_df['insights'] = merged_df['insights'].apply(
    lambda x: x if isinstance(x, dict) else default_insights
)

print(f"Insights generated in {time.time()-insights_start:.2f} seconds")

# 5. Product Ranking
print("\nCalculating product scores...")
ranking_start = time.time()

# Enhanced scoring formula
merged_df['score'] = (
    merged_df['average_rating'] *
    np.log1p(merged_df['rating_number']) *
    (1 + merged_df['sentiment_score']/10)
)

def get_recommendations(df, top_n=3):
    """Get recommendations with validation checks"""
    if len(df) < top_n + 1:
        return None

    # Get top products with minimum reviews
    top = df[df['rating_number'] >= 2].nlargest(top_n, 'score')
    if len(top) < top_n:
        top = df.nlargest(top_n, 'score')

    # Get worst product with minimum reviews
    worst = df[df['rating_number'] >= 1].nsmallest(1, 'score')
    if len(worst) == 0:
        worst = df.nsmallest(1, 'score')

    return pd.concat([top, worst])

recommendations = merged_df.groupby('main_category', group_keys=False).apply(
    get_recommendations).dropna()

print(f"Ranking completed in {time.time()-ranking_start:.2f} seconds")
print(f"Generated recommendations for {len(recommendations['main_category'].unique())} categories")

# 6. Article Generation
print("\nGenerating category articles...")
article_start = time.time()

def prepare_article_data(products):
    """Prepare structured data for article generation"""
    return [
        {
            'title': row['title'],
            'rating': row['average_rating'],
            'reviews': row['rating_number'],
            'praise': row['insights']['praises'][0] if row['insights']['praises'] else "No praises",
            'complaint': row['insights']['complaints'][0] if row['insights']['complaints'] else "No complaints",
            'is_top': i < (len(products) - 1)
        }
        for i, (_, row) in enumerate(products.iterrows())
    ]

def generate_gpt_article(category, products):
    """Generate article using GPT"""
    product_data = prepare_article_data(products)

    top_products = [p for p in product_data if p['is_top']]
    worst_product = [p for p in product_data if not p['is_top']][0]

    # Prepare prompt parts separately
    top_products_text = '\n'.join(
        f"- {p['title']} (★{p['rating']:.1f}, {p['reviews']} reviews)"
        for p in top_products
    )
    top_strengths = ', '.join(p['praise'] for p in top_products)

    prompt = f"""Analyze these products in the {category} category:

Top Products:
{top_products_text}

Product to Avoid:
- {worst_product['title']} (★{worst_product['rating']:.1f})

Key Insights:
{', '.join(f"{p['title']}: {p['praise']}" for p in top_products)}

Write a 3-paragraph comparison:
1. Overall category trends
2. Key differences between top products
3. Why the worst product underperforms"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo-0125",
            messages=[
                {"role": "system", "content": "You are an expert product analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.4,
            max_tokens=600,
            timeout=15
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"GPT Error: {str(e)}")
        return None

def generate_local_article(category, products):
    """Fallback article generation"""
    product_data = prepare_article_data(products)
    top_products = [p for p in product_data if p['is_top']]
    worst_product = [p for p in product_data if not p['is_top']][0]

    article = [
        f"# {category} Product Analysis",
        "\n## Top Products:",
        *[f"- {p['title']} (★{p['rating']:.1f}) - {p['praise']}" for p in top_products],
        "\n## Product to Avoid:",
        f"- {worst_product['title']} (★{worst_product['rating']:.1f}) - {worst_product['complaint']}",
        "\n## Summary:",
        f"The {category} category features {len(top_products)} top performers.",
        f"Avoid {worst_product['title']} due to consistent negative feedback."
    ]
    return '\n'.join(article)

# Generate and save articles
output_dir = "/content/drive/My Drive/Product_Recommendations"
os.makedirs(output_dir, exist_ok=True)

success_count = 0
for category, group in recommendations.groupby('main_category'):
    try:
        print(f"\nProcessing {category}...")

        # Try GPT first
        article = generate_gpt_article(category, group)

        # Fallback to local if needed
        if not article:
            article = generate_local_article(category, group)

        # Save to file
        filename = f"{category.replace('/', '_')}_recommendations.txt"
        with open(os.path.join(output_dir, filename), "w", encoding='utf-8') as f:
            f.write(article)

        success_count += 1
        print(f"Successfully generated {category}")
    except Exception as e:
        print(f"Failed to generate {category}: {str(e)}")

total_time = time.time() - article_start
print(f"\nArticle generation completed in {total_time:.2f} seconds")
print(f"Successfully generated {success_count} category reports")

# Save processed data
merged_df.to_csv(os.path.join(output_dir, "processed_products.csv"), index=False)
print("\nProcessing complete! Saved all results.")

Initializing services...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Services initialized in 4.06 seconds

Loading and sampling data...


NameError: name 'merged_df' is not defined

In [13]:
print("Review columns:", df_reviews.columns.tolist())


Review columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


In [16]:
print(analyzed_df.columns.tolist())


['rating_review', 'title_x', 'text', 'images_x', 'product_asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'review_date', 'main_category', 'title_y', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images_y', 'videos', 'store', 'categories', 'details', 'bought_together', 'subtitle', 'author', 'sentiment', 'sentiment_label', 'sentiment_score']


In [30]:
import pandas as pd
import numpy as np
from openai import OpenAI
from transformers import pipeline
from concurrent.futures import ThreadPoolExecutor
import time
import os
from datetime import datetime
import gzip
import json
from google.colab import userdata

# Configuration
CONFIG = {
    "openai_api_key": userdata.get("OPENAI_API_KEY"),
    "sample_size": 50000,
    "min_reviews": 5,
    "top_n": 3,
    "output_dir": "./product_articles"
}

# Load JSONL GZ files
def load_amazon_json_sample(filepath, max_lines=100000):
    data = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            data.append(json.loads(line))
    return pd.DataFrame(data)

# 1. Merge Reviews + Metadata
def load_and_merge(reviews, metadata):
    print("Merging datasets...")
    reviews = reviews.copy()
    metadata = metadata.copy()

    reviews['review_date'] = pd.to_datetime(reviews['timestamp'], unit='s', errors='coerce')
    reviews = reviews.rename(columns={'asin': 'product_asin', 'rating': 'rating_review'})
    metadata = metadata.rename(columns={'asin': 'product_asin'})

    merged = pd.merge(reviews, metadata, on='parent_asin', how='left')
    review_counts = merged['parent_asin'].value_counts()
    valid_products = review_counts[review_counts >= CONFIG['min_reviews']].index
    merged = merged[merged['parent_asin'].isin(valid_products)]

    return merged.sample(min(CONFIG['sample_size'], len(merged)), random_state=42)

# 2. Review Sentiment Analyzer
class ReviewAnalyzer:
    def __init__(self):
        self.sentiment = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=0,
            batch_size=64
        )

    def analyze_batch(self, texts):
        try:
            results = self.sentiment(list(texts))
            return [{'label': r['label'], 'score': r['score']} for r in results]
        except Exception:
            return [{'label': 'NEUTRAL', 'score': 0.5} for _ in texts]

    def process_reviews(self, df):
        print("Analyzing review sentiments...")
        start = time.time()

        with ThreadPoolExecutor(max_workers=4) as executor:
            batches = np.array_split(df['text'].fillna('').astype(str), 10)
            results = list(executor.map(self.analyze_batch, batches))

        df['sentiment'] = [item for sublist in results for item in sublist]
        df['sentiment_label'] = df['sentiment'].apply(lambda x: x['label'])
        df['sentiment_score'] = df['sentiment'].apply(lambda x: x['score'])

        print(f"Analyzed {len(df)} reviews in {time.time() - start:.2f}s")
        return df

# 3. Product Ranking
def rank_products(analyzed_df):
    print("Ranking products...")

    product_stats = analyzed_df.groupby(['parent_asin', 'main_category']).agg({
        'rating_review': ['mean', 'count'],
        'sentiment_label': lambda x: (x == 'POSITIVE').mean(),
        'title_y': 'first',
        'features': 'first',
        'price': 'mean'
    }).reset_index()

    product_stats.columns = [
        'parent_asin', 'main_category', 'avg_rating',
        'review_count', 'positive_ratio', 'product_title',
        'features', 'price'
    ]

    product_stats['score'] = (
        0.5 * product_stats['avg_rating'] +
        0.3 * np.log1p(product_stats['review_count']) +
        0.2 * product_stats['positive_ratio']
    )

    return product_stats

# 4. Article Generator
def structured_article_output(category, top_products, worst_product):
    lines = []
    lines.append("=" * 60)
    lines.append(f"{category.upper()}")
    lines.append("=" * 60)
    lines.append("**Top 3 Products:**\n")

    for i, p in enumerate(top_products, 1):
        lines.append(f"**{i}. {p['title']}**")
        lines.append(f"   - ⭐ Rating: {p['rating']:.1f} ({p['review_count']} reviews)")
        lines.append(f"   - 💲 Price: {p['price']}")
        lines.append(f"   - 👎 Complaints: {p['complaints']}\n")

    lines.append("**Key Differences:**")
    lines.append("- Add detailed comparison of features, suitability, etc.\n")

    lines.append("**Product to Avoid:**")
    lines.append(f"- **{worst_product['product_title']}**")
    lines.append(f"  - ⭐ Rating: {worst_product['avg_rating']:.1f}")
    lines.append(f"  - Reason: Lowest rating or repeated complaints\n")

    return "\n".join(lines)

class ArticleGenerator:
    def __init__(self):
        self.client = OpenAI(api_key=CONFIG["openai_api_key"])

    def _extract_key_complaints(self, product_df):
        neg_reviews = product_df[
            (product_df['sentiment_label'] == 'NEGATIVE') &
            (product_df['sentiment_score'] > 0.8)
        ]
        if len(neg_reviews) == 0:
            return ["No significant complaints"]
        top_complaints = neg_reviews['text'].str.lower().str.findall(
            r'\b(?:poor|bad|broken|issue|problem|defective)\b'
        ).explode().value_counts().head(3).index.tolist()
        return top_complaints or ["Some quality concerns reported"]

    def generate_article(self, category, products, all_reviews):
        try:
            if len(products) < 2:
                return f"⚠️ Not enough products in '{category}' to generate an article."

            top_products = products.nlargest(CONFIG['top_n'], 'score')
            worst_product = products.nsmallest(1, 'score').iloc[0]

            if worst_product['parent_asin'] in top_products['parent_asin'].values:
                return f"⚠️ Only one product available in '{category}'. Skipping duplicate output."

            product_details = []
            for _, product in top_products.iterrows():
                product_reviews = all_reviews[all_reviews['parent_asin'] == product['parent_asin']]
                complaints = self._extract_key_complaints(product_reviews)

                product_details.append({
                    'title': product['product_title'],
                    'rating': product['avg_rating'],
                    'review_count': product['review_count'],
                    'price': f"${product['price']:.2f}" if not pd.isna(product['price']) else "Price not available",
                    'complaints': ", ".join(complaints)
                })

            return structured_article_output(category, product_details, worst_product)

        except Exception as e:
            print(f"API Error for {category}: {str(e)}")
            return "⚠️ Failed to generate article due to error."

# 5. Main Execution
if __name__ == "__main__":
    reviews_path = "/content/drive/My Drive/Electronics.jsonl (1).gz"
    metadata_path = "/content/drive/My Drive/meta_Electronics.jsonl (1).gz"

    df_reviews = load_amazon_json_sample(reviews_path, max_lines=100000)
    df_metadata = load_amazon_json_sample(metadata_path, max_lines=100000)

    merged_df = load_and_merge(df_reviews, df_metadata)
    analyzer = ReviewAnalyzer()
    analyzed_df = analyzer.process_reviews(merged_df)
    ranked_products = rank_products(analyzed_df)

    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    generator = ArticleGenerator()

    for category, products in ranked_products.groupby('main_category'):
        print(f"\nProcessing {category}...")
        article = generator.generate_article(category, products, analyzed_df)

        if article.startswith("⚠️"):
            print(article)
            continue

        print(f"\n====== {category.upper()} ======\n")
        print(article)

        filename = f"{category.replace('/', '_')}_{datetime.now().strftime('%Y%m%d')}.md"
        with open(os.path.join(CONFIG['output_dir'], filename), 'w', encoding='utf-8') as f:
            f.write(article)

        print(f"Saved article for {category}")


Merging datasets...


Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)
Token indices sequence length is longer than the specified maximum sequence length for this model (536 > 512). Running this sequence through the model will result in indexing errors


Analyzing review sentiments...
Analyzed 26000 reviews in 0.53s
Ranking products...

Processing AMAZON FASHION...

====== AMAZON FASHION ======

AMAZON FASHION
**Top 3 Products:**

**1. Michael Kors Access Gen 4 MKGO Smartwatch- Lightweight Touchscreen Powered with Wear OS by Google with Heart Rate, GPS, NFC, and Smartphone Notifications**
   - ⭐ Rating: 4.3 (9 reviews)
   - 💲 Price: $244.18
   - 👎 Complaints: No significant complaints

**2. LOVEVOOK Geometric Luminous Purses and Handbags for Women Holographic Reflective Crossbody Bag Wallet**
   - ⭐ Rating: 4.4 (5 reviews)
   - 💲 Price: $37.99
   - 👎 Complaints: No significant complaints

**3. KAUKKO Vintage Casual Canvas and Leather Rucksack Backpack(FP702-1-COFFEE)**
   - ⭐ Rating: 4.2 (5 reviews)
   - 💲 Price: $42.99
   - 👎 Complaints: No significant complaints

**Key Differences:**
- Add detailed comparison of features, suitability, etc.

**Product to Avoid:**
- **Messenger Bag for Laptop Vintage Canvas Leather Crossbody Satchel Sh

In [27]:
def structured_article_output(category, top_products, worst_product):
    lines = []
    lines.append("=" * 60)
    lines.append(f"{category.upper()}")
    lines.append("=" * 60)
    lines.append("**Top 3 Products:**\n")

    for i, p in enumerate(top_products, 1):
        lines.append(f"**{i}. {p['title']}**")
        lines.append(f"   - ⭐ Rating: {p['rating']:.1f} ({p['review_count']} reviews)")
        lines.append(f"   - 💲 Price: {p['price']}")
        lines.append(f"   - 👎 Complaints: {p['complaints']}\n")

    lines.append("**Key Differences:**")
    lines.append("- Add detailed comparison of features, suitability, etc.\n")

    lines.append("**Product to Avoid:**")
    lines.append(f"- **{worst_product['product_title']}**")
    lines.append(f"  - ⭐ Rating: {worst_product['avg_rating']:.1f}")
    lines.append(f"  - Reason: Lowest rating or repeated complaints\n")

    return "\n".join(lines)


In [29]:
print(f"\n====== {category.upper()} ======\n")
print(article)



====== VIDEO GAMES ======

VIDEO GAMES
**Top 3 Products:**

**1. Turtle Beach Stealth 600 Gen 2 Wireless Gaming Headset for PS5, PS4, PS4 Pro, PlayStation, & Nintendo Switch with 50mm Speakers, 15-Hour Battery life, Flip-to-Mute Mic, and Spatial Audio - Black**
   - ⭐ Rating: 3.8 (5 reviews)
   - 💲 Price: $99.90
   - 👎 Complaints: No significant complaints

**Key Differences:**
- Add detailed comparison of features, suitability, etc.

**Product to Avoid:**
- **Turtle Beach Stealth 600 Gen 2 Wireless Gaming Headset for PS5, PS4, PS4 Pro, PlayStation, & Nintendo Switch with 50mm Speakers, 15-Hour Battery life, Flip-to-Mute Mic, and Spatial Audio - Black**
  - ⭐ Rating: 3.8
  - Reason: Lowest rating or repeated complaints

